# OneVoice V2 — English construction ASR
Clone source from GitHub. Use only an English-audio manifest with real speaker IDs and at least six voices.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['MODELSCOPE_CACHE'] = str(DRIVE_ROOT / 'model_cache/modelscope')
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'PyYAML', 'soundfile', 'librosa', 'funasr_onnx', 'modelscope'], check=True)
MANIFEST = DRIVE_ROOT / 'onevoice_audio_v2_1/manifest.jsonl'
REPORT_ROOT = DRIVE_ROOT / 'reports/en_asr'
print('Source:', REPO, '| Data:', MANIFEST, '| Reports:', REPORT_ROOT)


In [ ]:
subprocess.run([sys.executable, 'scripts/audit_audio_dataset.py', str(MANIFEST), '--language', 'en', '--min-speakers', '6', '--require-realized-snr', '--expected-clean', '8064', '--expected-noisy', '16128', '--report-dir', str(REPORT_ROOT / 'audit')], check=True)
for audio in ('clean', 'noisy'):
    subprocess.run([sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction', 'en2vi', '--split', 'test', '--audio', audio, '--denoiser', 'passthrough', '--report-dir', str(REPORT_ROOT / audio)], check=True)


In [ ]:
import json
{audio: json.loads((REPORT_ROOT / audio / 'aggregate.json').read_text(encoding='utf-8')) for audio in ('clean','noisy')}
